In [2]:
# ============================================================
# 🔥 CASTING DEFECT DETECTOR — COMPLETE ERROR-FREE COLAB CODE
# ============================================================

import os
import zipfile
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TensorFlow Version:", tf.__version__)

# ============================================================
# 1️⃣ Locate ZIP file automatically
# ============================================================

ZIP_NAME = "archive (1).zip"
ZIP_PATH = f"/content/{ZIP_NAME}"

if not os.path.exists(ZIP_PATH):
    raise Exception("❌ ZIP file not uploaded. Upload archive (1).zip!")

print("📦 ZIP File Found:", ZIP_PATH)

# ============================================================
# 2️⃣ Extract dataset
# ============================================================

EXTRACT_DIR = "/content"

print("⏳ Extracting ZIP...")
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

print("✅ Extraction complete!")

# ============================================================
# 3️⃣ Detect dataset folder
# ============================================================

possible_folders = [
    "/content/casting_data/casting_data",
    "/content/casting_data",
    "/content/casting_512x512",
]

DATASET_FOUND = None
for path in possible_folders:
    if os.path.exists(path):
        DATASET_FOUND = path
        break

if DATASET_FOUND is None:
    raise Exception("❌ Dataset not found. Check extracted folders!")

print("📂 Dataset detected at:", DATASET_FOUND)

train_dir = os.path.join(DATASET_FOUND, "train")
test_dir = os.path.join(DATASET_FOUND, "test")

# ============================================================
# 4️⃣ Image Preprocessing
# ============================================================

IMG_SIZE = 224
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1/255.,
    rotation_range=10,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
).flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_gen = ImageDataGenerator(rescale=1/255.).flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# ============================================================
# 5️⃣ Build CNN Model
# ============================================================

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

# ============================================================
# 6️⃣ Callbacks
# ============================================================

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.3, verbose=1),
    ModelCheckpoint("best_model.h5", save_best_only=True)
]

# ============================================================
# 7️⃣ Training
# ============================================================

history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=20,
    callbacks=callbacks
)

# ============================================================
# 8️⃣ Evaluation
# ============================================================

print("\n🔍 Evaluating Model...")
loss, acc = model.evaluate(test_gen)
print(f"✅ Test Accuracy: {acc:.4f}")

# ============================================================
# 9️⃣ Save Models for HuggingFace
# ============================================================

model.save("casting_defect_detector.h5")
model.save("model.keras")

print("\n🎉 All Done!")
print("📁 Saved files:")
print(" - casting_defect_detector.h5")
print(" - model.keras")
print(" - best_model.h5")


TensorFlow Version: 2.19.0
📦 ZIP File Found: /content/archive (1).zip
⏳ Extracting ZIP...
✅ Extraction complete!
📂 Dataset detected at: /content/casting_data/casting_data
Found 6633 images belonging to 2 classes.
Found 715 images belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

...

208/208 ━━━━━━━━━━━━━━━━━━━━ 80s 385ms/step - accuracy: 0.8902 - loss: 0.2697 - val_accuracy: 0.8923 - val_loss: 0.2072 - learning_rate: 0.0010
Epoch 7/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 378ms/step - accuracy: 0.9072 - loss: 0.2270 - val_accuracy: 0.7930 - val_loss: 0.4395 - learning_rate: 0.0010
Epoch 8/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 379ms/step - accuracy: 0.9147 - loss: 0.2113 - val_accuracy: 0.8629 - val_loss: 0.2610 - learning_rate: 0.0010
Epoch 9/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 378ms/step - accuracy: 0.9248 - loss: 0.1961

208/208 ━━━━━━━━━━━━━━━━━━━━ 80s 384ms/step - accuracy: 0.9248 - loss: 0.1960 - val_accuracy: 0.9259 - val_loss: 0.1838 - learning_rate: 0.0010
Epoch 10/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 377ms/step - accuracy: 0.9440 - loss: 0.1526 - val_accuracy: 0.8000 - val_loss: 0.4453 - learning_rate: 0.0010
Epoch 11/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step - accuracy: 0.9277 - loss: 0.1827

208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 379ms/step - accuracy: 0.9277 - loss: 0.1826 - val_accuracy: 0.9776 - val_loss: 0.0847 - learning_rate: 0.0010
Epoch 12/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 379ms/step - accuracy: 0.9462 - loss: 0.1382 - val_accuracy: 0.8839 - val_loss: 0.2541 - learning_rate: 0.0010
Epoch 13/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step - accuracy: 0.9532 - loss: 0.1233

208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 381ms/step - accuracy: 0.9532 - loss: 0.1233 - val_accuracy: 0.9748 - val_loss: 0.0816 - learning_rate: 0.0010
Epoch 14/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 78s 377ms/step - accuracy: 0.9686 - loss: 0.0895 - val_accuracy: 0.9469 - val_loss: 0.1534 - learning_rate: 0.0010
Epoch 15/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 377ms/step - accuracy: 0.9672 - loss: 0.0937 - val_accuracy: 0.9692 - val_loss: 0.0976 - learning_rate: 0.0010
Epoch 16/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 372ms/step - accuracy: 0.9575 - loss: 0.1189
Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0003000000142492354.
208/208 ━━━━━━━━━━━━━━━━━━━━ 78s 377ms/step - accuracy: 0.9575 - loss: 0.1189 - val_accuracy: 0.8350 - val_loss: 0.4308 - learning_rate: 0.0010
Epoch 17/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 79s 378ms/step - accuracy: 0.9700 - loss: 0.0902 - val_accuracy: 0.9580 - val_loss: 0.1299 - learning_rate: 3.0000e-04
Epoch 18/20
208/208 ━━━━━━━━━━━━━━━━━━━━ 78s 377ms/step - accuracy: 0.9810

✅ Test Accuracy: 0.9748

🎉 All Done!
📁 Saved files:
 - casting_defect_detector.h5
 - model.keras
 - best_model.h5
